In [ ]:
import pandas as pd
import pm4py
from pathlib import Path

# ============================================================
# CSV -> XES
# Input : ./mimicel_train.csv
# Output: ./results/mimicel.xes
# ============================================================

INPUT_CSV = Path("./../../MIMICEL_data/mimicel_train.csv")

OUTPUT_DIR = Path("./results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_XES = OUTPUT_DIR / "mimicel.xes"

CASE_COL = "stay_id"
ACT_COL = "activity"
TIME_COL = "timestamps"

print(f"[INFO] Loading CSV: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

required_cols = [CASE_COL, ACT_COL, TIME_COL]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

if df[TIME_COL].isna().any():
    raise ValueError(f"Invalid timestamps found: {df[TIME_COL].isna().sum()} rows")

df = df.rename(
    columns={
        CASE_COL: "case:concept:name",
        ACT_COL: "concept:name",
        TIME_COL: "time:timestamp",
    }
)

# PM4Py는 case id가 반드시 string이어야 함
df["case:concept:name"] = df["case:concept:name"].astype(str)

# activity도 string으로 맞추는 게 안전함
df["concept:name"] = df["concept:name"].astype(str)

df = df.sort_values(
    ["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

log = pm4py.convert_to_event_log(
    df,
    case_id_key="case:concept:name"
)

pm4py.write_xes(log, str(OUTPUT_XES))

print("=" * 60)
print("[SUCCESS] Saved XES")
print(f"Path : {OUTPUT_XES}")
print(f"Rows : {len(df):,}")
print(f"Cases: {df['case:concept:name'].nunique():,}")
print("=" * 60)

[INFO] Loading CSV: ..\..\MIMICEL_data\mimicel_train.csv


c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\pm4py\utils.py:991: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")


exporting log, completed traces ::   0%|          | 0/899 [00:00<?, ?it/s]

[SUCCESS] Saved XES
Path : xes_results\mimicel.xes
Rows : 16,176
Cases: 899
